<a href="https://colab.research.google.com/github/K-miy/420-a57-sf/blob/main/Donn%C3%A9es_synth%C3%A9tiques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nous allons utiliser le jeu de données "Adult Income Dataset" pour gérérer des données synthétiques avec une combinaison d'auto encodeur de type Beta-VAE (pour la dépendance) et de méthode inverse.

Lien vers les données et la documentation: https://archive.ics.uci.edu/dataset/2/adult



In [ ]:
# Install required libraries (for Google Colab)
!pip install tensorflow numpy pandas matplotlib scikit-learn



In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Dense, Lambda, Layer
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import keras.ops as ops
import seaborn as sns

In [ ]:
# 1. Load and Preprocess the Adult Income Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status',
           'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss',
           'hours_per_week', 'native_country', 'income']

# Load dataset
data = pd.read_csv(url, names=columns, na_values=" ?", skipinitialspace=True)
data.dropna(inplace=True)  # Drop missing values

data['income'] = data['income'].apply(lambda x: 1 if x == '>50K' else 0)  # Convert target to binary

data.head(10)


In [ ]:
# Separate columns
categorical_columns = ['workclass', 'education', 'marital_status', 'occupation',
                       'relationship', 'race', 'sex', 'native_country', 'income',
                       'age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

#continuous_columns = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

# Preprocess and encode Continuous Variables (Scale [0, 1])
#scaler_cont = MinMaxScaler()
#data[continuous_columns] = scaler_cont.fit_transform(data[continuous_columns])

# Preprocess Categorical Variables using Empirical CDF (Monte Carlo-inspired)
encoders = {}
categorical_mappings = {}

data2 = data.copy()

# Encode categorical data as uniform [0, 1] based on empirical CDF
for col in categorical_columns:
    unique_vals = data[col].unique()
    encoders[col] = unique_vals  # Save encoder mapping

    # Compute empirical CDF ranks scaled to [0, 1]
    mapping = {val: (i + 1) / len(unique_vals) for i, val in enumerate(sorted(unique_vals))}
    categorical_mappings[col] = mapping  # Save mapping

    # Map values to CDF-based ranks
    data2[col] = data2[col].map(mapping)

data2.head(50)

In [ ]:
# Separate features and target (prep pour le VEE auto-encoder)
X = data2.drop(columns=['income'])
y = data2['income']
X['income'] = y  # Include target in VAE training to capture dependencies

X.head(10)
#y.head(10)

In [ ]:
# 2. Build Variational Auto-Encoder (VAE)
latent_dim = 64  # Latent space dimension

# Encoder
inputs = Input(shape=(X.shape[1],))
h = Dense(128, activation='relu')(inputs)
h2 = Dense(128, activation='relu')(h)
z_mean = Dense(latent_dim)(h2)
z_log_var = Dense(latent_dim)(h2)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling, output_shape=(latent_dim,))([z_mean, z_log_var])

# Decoder
decoder_h2 = Dense(128, activation='relu')
decoder_h = Dense(128, activation='relu')
decoder_outputs = Dense(X.shape[1], activation='sigmoid')

h2_decoded = decoder_h2(z)
h_decoded = decoder_h(h2_decoded)
outputs = decoder_outputs(h_decoded)

# VAE Model
vae = Model(inputs, outputs)

# Define a custom loss function: Reconstruction Loss (=fonction de perte classique) + KL Divergence (pénalité pour que les Z (l'encodeur) soit une loi Normale multivariée)

beta = 0.05 # Hyper-parameter to limit the effect of the divergence KL in the global loss function

class VAELossLayer(Layer):
    def __init__(self, **kwargs):
        super(VAELossLayer, self).__init__(**kwargs)

    def call(self, inputs, outputs, z_mean, z_log_var):
        # Reconstruction loss (mean squared error)
        reconstruction_loss = ops.mean(ops.square(inputs - outputs))
        # KL Divergence loss
        kl_loss = -0.05 * ops.mean(1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
        # Combine both losses
        loss = 1*reconstruction_loss + beta*kl_loss
        self.add_loss(loss)
        return outputs # We still need to return the outputs.

# Add the custom loss function to the VEE model as a layer
loss_layer = VAELossLayer()
outputs = loss_layer(inputs,outputs,z_mean,z_log_var)

# Compile the model, the loss is now handled by the loss layer
vae = Model(inputs,outputs)
vae.compile(optimizer='adam')

In [ ]:
# 3. Train VAE
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
vae.fit(X_train, X_train, epochs=50, batch_size=64, validation_data=(X_test, X_test))


In [ ]:
# 4. Generate Synthetic Data (forme encodée)

# Create the generator
decoder_input = Input(shape=(latent_dim,))
h2_decoded = decoder_h2(decoder_input)
h_decoded = decoder_h(h2_decoded)
synthetic_outputs = decoder_outputs(h_decoded)

generator = Model(decoder_input, synthetic_outputs)

# Make the simulations
z_sample = np.random.normal(size=(len(X), latent_dim))  # Sample latent space
synthetic_data = generator.predict(z_sample)

# Reverse Scaling and Encoding for Synthetic Data
synthetic_df = pd.DataFrame(synthetic_data, columns=X.columns)

synthetic_df[0:50]

#z_sample[0:50]

In [ ]:
# Reverse scaling for continuous columns
synthetic_df2 = synthetic_df.copy()
#synthetic_df2[continuous_columns] = scaler_cont.inverse_transform(synthetic_df2[continuous_columns])

# Reverse empirical CDF for categorical columns
for col in categorical_columns:
    # Map back using percentiles
    inverse_mapping = {v: k for k, v in categorical_mappings[col].items()}
    synthetic_df2[col] = synthetic_df2[col].apply(lambda x: inverse_mapping[min(inverse_mapping.keys(), key=lambda k: abs(k-x))])

#synthetic_df.head(50)
synthetic_df2.head(50)

In [ ]:
# 5. Validation: Logistic Regression Coefficients
X_original = data2.drop(columns=['income'])
y_original = data2['income'].astype(int)

X_synthetic = synthetic_df.drop(columns=['income'])
y_synthetic = synthetic_df2['income']

# Re-encode synthetic categorical features to numeric values
#for col in categorical_columns:
#    if col != 'income': # Skip the income column
#        X_synthetic[col] = X_synthetic[col].map(categorical_mappings[col])

# Train Logistic Regression Models
lr_original = LogisticRegression(max_iter=5000)
lr_original.fit(X_original, y_original)
original_coeff = lr_original.coef_[0]

lr_synthetic = LogisticRegression(max_iter=5000)
lr_synthetic.fit(X_synthetic, y_synthetic)
synthetic_coeff = lr_synthetic.coef_[0]

# Compare coefficients
coeff_df = pd.DataFrame({'Original': original_coeff, 'Synthetic': synthetic_coeff}, index=X_original.columns)
print("\nLogistic Regression Coefficients Comparison:")
print(coeff_df)

# 6. Visualization and Comparison
fig, axes = plt.subplots(5, 3, figsize=(15, 25))
for i, col in enumerate(['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status',
           'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss',
           'hours_per_week', 'native_country', 'income']):
    ax = axes[i // 3, i % 3]
    ax.hist(data2[col], bins=20, alpha=0.5, label='Original')
    ax.hist(synthetic_df[col], bins=20, alpha=0.5, label='Synthetic')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()


# Display samples
print("\nOriginal Data Sample:")
print(data.head(5))

print("\nSynthetic Data Sample:")
print(synthetic_df.head(5))






In [ ]:
# 7. Compare the projections obtained with original vs synthetic data
orig_pred = lr_original.predict(X_original)
synth_pred = lr_synthetic.predict(X_original)

print(orig_pred[0:50])
print(synth_pred[0:50])

cm = confusion_matrix(orig_pred,synth_pred)
cm

In [ ]:
#8. Compare the bi-variate distributions

# Generic function to create bi-variate graphs
def validate_bivariate_distributions(original_df, synthetic_df, columns_to_compare):
    """
    Validate bivariate distributions between original and synthetic data.

    Args:
        original_df (pd.DataFrame): Original dataset.
        synthetic_df (pd.DataFrame): Synthetic dataset.
        columns_to_compare (list of tuples): Pairs of columns to compare.
    """
    num_pairs = len(columns_to_compare)
    fig, axes = plt.subplots(num_pairs, 2, figsize=(10, 5 * num_pairs))

    for i, (col1, col2) in enumerate(columns_to_compare):
        # Original Data Plot
        sns.kdeplot(
            data=original_df, x=col1, y=col2, fill=True, cmap="Blues", ax=axes[i, 0]
        )
        axes[i, 0].set_title(f"Original: {col1} vs {col2}")

        # Synthetic Data Plot
        sns.kdeplot(
            data=synthetic_df, x=col1, y=col2, fill=True, cmap="Oranges", ax=axes[i, 1]
        )
        axes[i, 1].set_title(f"Synthetic: {col1} vs {col2}")

    plt.tight_layout()
    plt.show()

# Variables to be compared 2 by 2
columns_to_compare = [('age', 'education'),('age','marital_status'),('age','income'),('occupation','sex'),('race','native_country')]

# Calling the function
validate_bivariate_distributions(data2, synthetic_df, columns_to_compare)
